# Gate-Gap Failure Attribution v1

目的：把近期 reward / model-selection / execution 诊断的 `selection.json` 聚合到同一个 gate-gap 视角，回答当前 overfit 风险下下一步应该优化什么。

目标门槛：超额收益 `>= 30pp`，MaxDD `<= 15%`，训练和选择必须继续受 walk-forward 证据约束。

本 notebook 是单次运行结论的主记录；CSV / JSON / Markdown 只是 sidecar evidence。

## External Basis

- [Microsoft Qlib GitHub](https://github.com/microsoft/qlib): 保持 qlib workflow 和可复现实验/回测边界。
- [Qlib arXiv paper](https://arxiv.org/abs/2009.11189): 使用 AI-oriented quantitative investment platform 的工作流视角组织数据、模型、回测和分析。
- [Deflated Sharpe Ratio / multiple testing](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551): 新实验必须防止多次筛选造成的过拟合。
- [Probability of Backtest Overfitting](https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf): 继续使用 walk-forward / PBO 风格证据，而不是只看单一窗口最优。

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.oos_diagnostics import (
    attribute_gate_gaps,
    normalise_gate_metrics_from_selection,
    render_gate_gap_review,
    summarize_gate_gap_attribution,
)

ROOT = Path.cwd()
CARD_DIR = ROOT / "runtime/model_selection/card/gate_gap_failure_attribution_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

selection_paths = {
    "state_conditioned_drawdown_rebound_reward_train_v1": ROOT / "runtime/reward/card/state_conditioned_drawdown_rebound_reward_train_v1/selection.json",
    "mode_conditioned_rebound_reward_train_v2": ROOT / "runtime/reward/card/mode_conditioned_rebound_reward_train_v2/selection.json",
    "drawdown_rebound_reward_train_v1": ROOT / "runtime/reward/card/drawdown_rebound_reward_train_v1/selection.json",
    "drawdown_aware_reward_proxy_v1": ROOT / "runtime/reward/card/drawdown_aware_reward_proxy_v1/selection.json",
    "drawdown_rebound_checkpoint_pbo_v1": ROOT / "runtime/model_selection/card/drawdown_rebound_checkpoint_pbo_v1/selection.json",
    "state_conditioned_actor_selector_v1": ROOT / "runtime/model_selection/card/state_conditioned_actor_selector_v1/selection.json",
    "barrier_drawdown_budget_overlay_v1": ROOT / "runtime/execution/card/barrier_drawdown_budget_overlay_v1/selection.json",
}

external_references = [
    {
        "label": "Microsoft Qlib",
        "url": "https://github.com/microsoft/qlib",
        "note": "repo workflow and reproducible quant platform boundary",
    },
    {
        "label": "Qlib arXiv",
        "url": "https://arxiv.org/abs/2009.11189",
        "note": "AI-oriented quantitative investment workflow",
    },
    {
        "label": "Deflated Sharpe Ratio",
        "url": "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
        "note": "multiple-testing / selection-bias control",
    },
    {
        "label": "Probability of Backtest Overfitting",
        "url": "https://www.davidhbailey.com/dhbpapers/backtest-overfitting.pdf",
        "note": "walk-forward and PBO-style anti-overfit guard",
    },
]


In [2]:
metrics_by_profile: dict[str, dict[str, dict[str, float]]] = {}
skipped: list[dict[str, str]] = []
coverage_rows: list[dict[str, object]] = []

for label, path in selection_paths.items():
    if not path.exists():
        skipped.append({"profile": label, "reason": "missing_selection", "path": str(path)})
        continue
    selection = json.loads(path.read_text(encoding="utf-8"))
    metrics = normalise_gate_metrics_from_selection(selection)
    if metrics:
        metrics_by_profile[label] = metrics
    else:
        skipped.append({"profile": label, "reason": "no_window_gate_metrics", "path": str(path)})
    coverage_rows.append(
        {
            "profile": label,
            "path": str(path.relative_to(ROOT)),
            "window_count": len(metrics),
            "windows": ",".join(sorted(metrics)),
        }
    )

coverage = pd.DataFrame(coverage_rows)
skipped_df = pd.DataFrame(skipped)
coverage.to_csv(CARD_DIR / "selection_metric_coverage.csv", index=False)
skipped_df.to_csv(CARD_DIR / "selection_metric_skipped.csv", index=False)

display(coverage)
if not skipped_df.empty:
    display(skipped_df)


,profile,path,window_count,windows
0,state_conditioned_drawdown_rebound_reward_trai...,runtime/reward/card/state_conditioned_drawdown...,4,"test_proxy,wf_2020,wf_2022,wf_2023_2024"
1,mode_conditioned_rebound_reward_train_v2,runtime/reward/card/mode_conditioned_rebound_r...,8,"test_2020,test_2022,test_2023_2024,test_proxy,..."
2,drawdown_rebound_reward_train_v1,runtime/reward/card/drawdown_rebound_reward_tr...,1,main
3,drawdown_aware_reward_proxy_v1,runtime/reward/card/drawdown_aware_reward_prox...,0,
4,drawdown_rebound_checkpoint_pbo_v1,runtime/model_selection/card/drawdown_rebound_...,4,"strict_oos,wf_2020,wf_2022,wf_2023_2024"
5,state_conditioned_actor_selector_v1,runtime/model_selection/card/state_conditioned...,4,"strict_oos,wf_2020,wf_2022,wf_2023_2024"
6,barrier_drawdown_budget_overlay_v1,runtime/execution/card/barrier_drawdown_budget...,0,


,profile,reason,path
0,drawdown_aware_reward_proxy_v1,no_window_gate_metrics,/Users/rziiiii/.openclaw/workspace/runtime/rew...
1,barrier_drawdown_budget_overlay_v1,no_window_gate_metrics,/Users/rziiiii/.openclaw/workspace/runtime/exe...


In [3]:
gate_gaps = attribute_gate_gaps(
    metrics_by_profile,
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
profile_summary = summarize_gate_gap_attribution(gate_gaps)

gate_gaps.to_csv(CARD_DIR / "gate_gap_attribution.csv", index=False)
profile_summary.to_csv(CARD_DIR / "gate_gap_profile_summary.csv", index=False)

display(profile_summary)
display(gate_gaps.head(30))


,profile,window_count,gate_pass_count,failure_count,dominant_failure_type,worst_window,max_severity_score,recommended_next_mechanism
0,mode_conditioned_rebound_reward_train_v2,8,0,8,joint_tradeoff,test_proxy,2429.814066,walk_forward_weighted_multi_objective_selection
1,drawdown_rebound_reward_train_v1,1,0,1,joint_tradeoff,main,2283.209141,walk_forward_weighted_multi_objective_selection
2,state_conditioned_drawdown_rebound_reward_trai...,4,0,4,joint_tradeoff,test_proxy,1785.683201,walk_forward_weighted_multi_objective_selection
3,drawdown_rebound_checkpoint_pbo_v1,4,0,4,drawdown_control,strict_oos,395.500189,drawdown_controller_or_sequence_risk_feature
4,state_conditioned_actor_selector_v1,4,0,4,drawdown_control,wf_2022,347.749068,drawdown_controller_or_sequence_risk_feature


,profile,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,relative_gap,max_dd_gap,relative_pass,maxdd_pass,gate_pass,failure_type,severity_score,primary_optimization_path
0,mode_conditioned_rebound_reward_train_v2,test_proxy,496.780712,2542.027442,-2107.471934,0.991448,44.234213,0.267829,2137.471934,29.234213,False,False,False,joint_tradeoff,2429.814066,walk_forward_weighted_multi_objective_selection
1,drawdown_rebound_reward_train_v1,main,392.580698,2542.027442,-2026.444285,1.026548,37.676486,0.281184,2056.444285,22.676486,False,False,False,joint_tradeoff,2283.209141,walk_forward_weighted_multi_objective_selection
2,state_conditioned_drawdown_rebound_reward_trai...,test_proxy,619.708826,2542.027442,-1470.032017,1.016311,43.565118,0.382524,1500.032017,28.565118,False,False,False,joint_tradeoff,1785.683201,walk_forward_weighted_multi_objective_selection
3,mode_conditioned_rebound_reward_train_v2,wf_2020,50.083327,355.736160,-304.710394,1.402969,23.086304,0.297235,334.710394,8.086304,False,False,False,joint_tradeoff,415.573434,walk_forward_weighted_multi_objective_selection
4,state_conditioned_drawdown_rebound_reward_trai...,wf_2020,56.775553,355.736160,-275.089294,1.412174,24.413070,0.431536,305.089294,9.413070,False,False,False,joint_tradeoff,399.219997,walk_forward_weighted_multi_objective_selection
5,mode_conditioned_rebound_reward_train_v2,test_2020,51.025766,355.736160,-304.710394,1.719022,21.434020,0.297235,334.710394,6.434020,False,False,False,joint_tradeoff,399.050595,walk_forward_weighted_multi_objective_selection
6,drawdown_rebound_checkpoint_pbo_v1,strict_oos,NaN,0.000000,610.302614,NaN,54.550019,0.000000,0.000000,39.550019,True,False,False,drawdown_control,395.500189,drawdown_controller_or_sequence_risk_feature
7,state_conditioned_actor_selector_v1,wf_2022,-47.263003,-86.588072,39.325069,-1.356866,49.774907,0.273316,0.000000,34.774907,True,False,False,drawdown_control,347.749068,drawdown_controller_or_sequence_risk_feature
8,mode_conditioned_rebound_reward_train_v2,test_2022,-40.753958,-76.614257,65.015154,-1.507673,43.909386,0.131381,0.000000,28.909386,True,False,False,drawdown_control,289.093862,drawdown_controller_or_sequence_risk_feature
9,state_conditioned_actor_selector_v1,strict_oos,1667.806381,816.512500,851.293881,1.151077,43.534970,0.449886,0.000000,28.534970,True,False,False,drawdown_control,285.349698,drawdown_controller_or_sequence_risk_feature


In [4]:
if gate_gaps.empty:
    mechanism_summary = pd.DataFrame()
else:
    mechanism_summary = (
        gate_gaps.groupby("failure_type", as_index=False)
        .agg(
            profiles=("profile", "nunique"),
            windows=("window", "count"),
            mean_relative_gap=("relative_gap", "mean"),
            max_relative_gap=("relative_gap", "max"),
            mean_max_dd_gap=("max_dd_gap", "mean"),
            max_max_dd_gap=("max_dd_gap", "max"),
            mean_avg_pos=("avg_pos", "mean"),
            max_severity=("severity_score", "max"),
        )
        .sort_values(["profiles", "max_severity"], ascending=[False, False])
        .reset_index(drop=True)
    )
mechanism_summary.to_csv(CARD_DIR / "mechanism_summary.csv", index=False)
display(mechanism_summary)


,failure_type,profiles,windows,mean_relative_gap,max_relative_gap,mean_max_dd_gap,max_max_dd_gap,mean_avg_pos,max_severity
0,joint_tradeoff,5,11,626.405482,2137.471934,14.874178,29.234213,0.307430,2429.814066
1,drawdown_control,4,8,0.000000,0.000000,23.072385,39.550019,0.169165,395.500189
2,near_miss_drawdown,2,2,0.000000,0.000000,0.565168,0.900697,0.152222,9.006975


In [5]:
poor_context_dir = ROOT / "runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate"
poor_cluster_path = poor_context_dir / "poor_window_clusters.csv"
feature_delta_path = poor_context_dir / "feature_delta_clusters.csv"

poor_clusters = pd.read_csv(poor_cluster_path) if poor_cluster_path.exists() else pd.DataFrame()
feature_deltas = pd.read_csv(feature_delta_path) if feature_delta_path.exists() else pd.DataFrame()

poor_clusters.head(10).to_csv(CARD_DIR / "poor_window_cluster_context.csv", index=False)
feature_deltas.head(15).to_csv(CARD_DIR / "feature_delta_context.csv", index=False)

display(poor_clusters.head(10))
display(feature_deltas.head(15))


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C002,2025-04-04,2025-07-22,5,30,"balanced_rule_residual_v19_reg,balanced_rule_r...",-160.502243,-199.393437,5.924443,16.324924,0.519568,166.426685,203.420490,underparticipation,reentry_participation_or_behavior_clone
1,C003,2026-02-04,2026-05-21,4,18,"balanced_rule_residual_v22,current_runtime_pap...",-167.300896,-198.889223,10.842927,24.118142,0.089418,178.143823,205.310696,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-18,2020-06-19,4,12,"balanced_rule_residual_v19_reg,balanced_rule_r...",-158.096497,-174.787005,8.683677,13.885509,0.514949,166.780175,182.311302,underparticipation,reentry_participation_or_behavior_clone


,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score
0,vol_of_vol_21,5,2.909053,2.909053,1.0,-0.029445,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.212323
1,tracking_error_vol_21,4,3.153269,3.153269,1.0,0.008294,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074991
2,decay_63,5,-2.831926,2.831926,1.0,0.065458,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074129
3,realized_vol_63,5,2.528318,2.528318,1.0,-0.015127,"balanced_rule_residual_v19_reg,balanced_rule_r...",4.530138
4,tracking_error_21,4,-2.161666,2.161666,1.0,0.090702,"balanced_rule_residual_v19_reg,balanced_rule_r...",3.479067
5,realized_vol_21,5,1.623577,1.623577,1.0,-0.003108,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.909059
6,vol_coef_21,5,1.464646,1.464646,1.0,-0.003431,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.624293
7,vix_level,4,1.270719,1.270719,1.0,0.097231,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.045143
8,jump_variation_5,5,1.125866,1.125866,1.0,0.005165,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.017282
9,realized_vol_10,5,1.111914,1.111914,1.0,-0.001980,"balanced_rule_residual_v19_reg,balanced_rule_r...",1.992282


In [6]:
latest_profile = "state_conditioned_drawdown_rebound_reward_train_v1"
latest_rows = gate_gaps[gate_gaps["profile"] == latest_profile].copy()
latest_summary = profile_summary[profile_summary["profile"] == latest_profile].copy()
latest_near_miss = latest_rows[latest_rows["failure_type"].str.contains("near_miss", na=False)]
latest_joint = latest_rows[latest_rows["failure_type"] == "joint_tradeoff"]
latest_drawdown = latest_rows[latest_rows["failure_type"] == "drawdown_control"]

if not latest_summary.empty:
    latest_dominant = str(latest_summary.iloc[0]["dominant_failure_type"])
    next_mechanism = str(latest_summary.iloc[0]["recommended_next_mechanism"])
else:
    latest_dominant = "missing"
    next_mechanism = "collect_gate_metrics"

recurring_clusters = [] if poor_clusters.empty else poor_clusters.head(3)["cluster_start"].astype(str).tolist()
feature_candidates = [] if feature_deltas.empty else feature_deltas.head(5)["feature"].astype(str).tolist()

conclusion_md = f"""## Runtime Conclusion

Decision: `diagnostic_no_promotion`. The latest state-conditioned reward branch is not a promotion candidate under +30pp / <=15% gates.

Key evidence:

- Latest branch dominant gate failure: `{latest_dominant}`; recommended mechanism from reusable attribution: `{next_mechanism}`.
- Latest branch has `{len(latest_near_miss)}` near-miss window(s), but also `{len(latest_joint)}` joint trade-off window(s) and `{len(latest_drawdown)}` drawdown-control window(s). The near miss on `wf_2023_2024` is not enough to offset `test_proxy`, `wf_2020`, and `wf_2022` failures.
- Across recent cards, repeated severe failures are split between `joint_tradeoff` and `drawdown_control`, so another stronger global reward penalty is not the next useful experiment.
- Prior poor-window aggregate still points to underparticipation clusters around `{', '.join(recurring_clusters)}`. That means the next training/selection loop must not solve drawdown by collapsing participation in rebound windows.
- Top feature-difference hypotheses to route into factor IC / single-signal / walk-forward validation, not direct promotion: `{', '.join(feature_candidates)}`.

Next valid loop:

1. Build a walk-forward-weighted multi-objective selector/training objective that scores +30pp excess and <=15% MaxDD jointly, not as a single scalar validation return.
2. Add state-conditioned participation/reentry pressure only in rebound/bull-recovery states, while keeping explicit drawdown control in 2022-style stress states.
3. Before retraining, run a factor notebook for the SOXX/SOXL spread and the recurring volatility/tracking-error candidates to decide whether they are features, gates, or only diagnostics.
"""

review = render_gate_gap_review(gate_gaps, profile_summary, external_references=external_references)
(CARD_DIR / "gate_gap_review.md").write_text(review, encoding="utf-8")
(CARD_DIR / "runtime_conclusion.md").write_text(conclusion_md, encoding="utf-8")

selection = {
    "experiment": "gate_gap_failure_attribution_v1",
    "decision": "diagnostic_no_promotion",
    "target": {
        "relative_return_min_pp": TARGET_RELATIVE_RETURN,
        "max_dd_max_pct": TARGET_MAX_DD,
    },
    "latest_profile": latest_profile,
    "latest_dominant_failure_type": latest_dominant,
    "recommended_next_mechanism": next_mechanism,
    "avoid": [
        "another stronger global reward penalty",
        "promotion based on wf_2023_2024 near miss only",
        "directly promoting feature deltas without IC and walk-forward checks",
    ],
    "next_valid_loop": [
        "walk-forward weighted multi-objective selector/training objective",
        "state-conditioned rebound participation with drawdown-state guard",
        "SOXX/SOXL spread factor notebook before feature promotion",
    ],
    "artifacts": {
        "notebook": str((CARD_DIR / "gate_gap_failure_attribution_v1.ipynb").relative_to(ROOT)),
        "coverage": str((CARD_DIR / "selection_metric_coverage.csv").relative_to(ROOT)),
        "skipped": str((CARD_DIR / "selection_metric_skipped.csv").relative_to(ROOT)),
        "gate_gaps": str((CARD_DIR / "gate_gap_attribution.csv").relative_to(ROOT)),
        "profile_summary": str((CARD_DIR / "gate_gap_profile_summary.csv").relative_to(ROOT)),
        "mechanism_summary": str((CARD_DIR / "mechanism_summary.csv").relative_to(ROOT)),
        "poor_window_cluster_context": str((CARD_DIR / "poor_window_cluster_context.csv").relative_to(ROOT)),
        "feature_delta_context": str((CARD_DIR / "feature_delta_context.csv").relative_to(ROOT)),
        "review": str((CARD_DIR / "gate_gap_review.md").relative_to(ROOT)),
        "runtime_conclusion": str((CARD_DIR / "runtime_conclusion.md").relative_to(ROOT)),
    },
    "external_references": external_references,
}
(CARD_DIR / "selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown(conclusion_md))


## Runtime Conclusion

Decision: `diagnostic_no_promotion`. The latest state-conditioned reward branch is not a promotion candidate under +30pp / <=15% gates.

Key evidence:

- Latest branch dominant gate failure: `joint_tradeoff`; recommended mechanism from reusable attribution: `walk_forward_weighted_multi_objective_selection`.
- Latest branch has `1` near-miss window(s), but also `2` joint trade-off window(s) and `1` drawdown-control window(s). The near miss on `wf_2023_2024` is not enough to offset `test_proxy`, `wf_2020`, and `wf_2022` failures.
- Across recent cards, repeated severe failures are split between `joint_tradeoff` and `drawdown_control`, so another stronger global reward penalty is not the next useful experiment.
- Prior poor-window aggregate still points to underparticipation clusters around `2025-04-04, 2026-02-04, 2020-03-18`. That means the next training/selection loop must not solve drawdown by collapsing participation in rebound windows.
- Top feature-difference hypotheses to route into factor IC / single-signal / walk-forward validation, not direct promotion: `vol_of_vol_21, tracking_error_vol_21, decay_63, realized_vol_63, tracking_error_21`.

Next valid loop:

1. Build a walk-forward-weighted multi-objective selector/training objective that scores +30pp excess and <=15% MaxDD jointly, not as a single scalar validation return.
2. Add state-conditioned participation/reentry pressure only in rebound/bull-recovery states, while keeping explicit drawdown control in 2022-style stress states.
3. Before retraining, run a factor notebook for the SOXX/SOXL spread and the recurring volatility/tracking-error candidates to decide whether they are features, gates, or only diagnostics.
